# Финальный ML dataset М1-М5

Цель ноутбука: собрать и проверить общий дневной датасет из `m1_features`, `m2_features`, `m3_features`, `m4_features`, `m5_features` без расчета финального LSI

Старая версия ноутбука считала PCA/IsolationForest и использовала старые M5-колонки `MAD_score_Budget`, `MAD_score_Liquidity`. В актуальном backend этих колонок нет: M5 теперь строится через лагированную ликвидность, budget с publication lag и признаки Росказны


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'backend').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.src.services.final_dataset_builder import build_final_ml_dataset
from backend.src.services.final_dataset_builder import save_csv
from backend.src.services.final_dataset_builder import save_parquet

PROCESSED = PROJECT_ROOT / 'data' / 'processed'


## Методология объединения

- Базовая дневная сетка берется из `m5_features`: это даты таблицы ликвидности банковского сектора ЦБ, то есть естественная ось будущего ML/backtest
- М1 присоединяется `merge_asof` по `averaging_period_end`, а не по началу периода, потому что часть признаков М1 использует информацию за весь период усреднения
- М2 и М3 присоединяются только по точной дате аукциона; forward fill для аукционов не используется
- М4 присоединяется по точной календарной дате
- M5 присоединяется по точной дате, потому что уже находится на дневной сетке ЦБ
- `tax_pressure_smoothed` и сглаженный `Seasonal_Factor` из M4 не включены в финальный dataset, потому что они считаются центрированным окном и могут использовать будущие дни; вместо этого берется `Seasonal_Factor_raw`


In [2]:
final_df = build_final_ml_dataset()
save_csv(final_df)
save_parquet(final_df)

print('Финальный датасет сохранен')
print('CSV:', PROCESSED / 'final_ml_dataset.csv')
print('Parquet:', PROCESSED / 'final_ml_dataset.parquet')
print('Форма:', final_df.shape)
print('Даты:', final_df['date'].min().date(), '->', final_df['date'].max().date())
print('Дубли дат:', int(final_df.duplicated('date').sum()))


Финальный датасет сохранен
CSV: /Users/iladroskov/Coding/mai/MathMod_2026/psb_case/data/processed/final_ml_dataset.csv
Parquet: /Users/iladroskov/Coding/mai/MathMod_2026/psb_case/data/processed/final_ml_dataset.parquet
Форма: (3077, 102)
Даты: 2014-02-03 -> 2026-05-08
Дубли дат: 0


In [3]:
module_counts = {
    prefix: sum(column.startswith(prefix + '_') for column in final_df.columns)
    for prefix in ['m1', 'm2', 'm3', 'm4', 'm5']
}
print('Колонки по модулям:', module_counts)

availability_cols = [
    'm1_features_available',
    'm2_auction_flag',
    'm3_auction_flag',
    'm3_history_available',
    'm4_calendar_available',
    'm5_features_available',
]
display(final_df[availability_cols].mean().rename('mean').to_frame())


Колонки по модулям: {'m1': 27, 'm2': 13, 'm3': 16, 'm4': 15, 'm5': 30}


,mean
m1_features_available,1.000000
m2_auction_flag,0.062398
m3_auction_flag,0.145596
m3_history_available,0.843679
m4_calendar_available,1.000000
m5_features_available,1.000000


In [4]:
nulls = final_df.isna().sum().sort_values(ascending=False)
display(nulls[nulls > 0].head(25).rename('null_rows').to_frame())

print('Строки М2 с аукционом:', int((final_df['m2_auction_flag'] == 1).sum()))
print('Строки М3 с аукционом:', int((final_df['m3_auction_flag'] == 1).sum()))
print('M1 latest available date:', final_df['m1_available_date'].max().date())


,null_rows
m2_cover_ratio,2885
m2_key_rate,2885
m2_rate_for_spread,2885
m2_rate_spread,2885
m3_yield_spread,2655
m3_weighted_yield,2654
m3_cover_ratio,2629
m5_roskazna_cover_ratio_lag_1d,2190
m5_roskazna_demand_volume_mln_rub_lag_1d,2190
m5_roskazna_bidders_count_lag_1d,2190


Строки М2 с аукционом: 192
Строки М3 с аукционом: 448
M1 latest available date: 2026-04-14


## Критика старой версии анализа

Проблема: использовались M5-колонки `MAD_score_Budget` и `MAD_score_Liquidity`, которых больше нет  
Почему важно: ноутбук не соответствует текущему backend и не учитывает Росказну  
Что сделано: финальный dataset теперь строится через актуальный `m5_features`

Проблема: M1 присоединялся по `date`, то есть по началу периода усреднения  
Почему важно: `ruonia_period_avg` и часть М1-сигнала известны только после окончания периода  
Что сделано: M1 присоединяется as-of по `averaging_period_end`

Проблема: в старом ноутбуке сразу обучался PCA/IsolationForest и считался LSI  
Почему важно: это отдельная методологическая задача; там были full-sample scaler/PCA/min-max, произвольный `contamination=0.06` и риск leakage при backtest  
Что сделано: текущий ноутбук только собирает и проверяет ML dataset

Проблема: M3 `MAD_score_cover` в старой логике шел с прямым знаком  
Почему важно: для ОФЗ низкий cover ratio означает стресс, высокий cover ratio скорее обратный сигнал  
Что сделано: добавлен `m3_cover_stress_score = -m3_MAD_score_cover`

Проблема: M4 сглаженные признаки рассчитаны центрированными окнами  
Почему важно: это может использовать будущие дни  
Что сделано: в финальный dataset включен `m4_Seasonal_Factor_raw`, а не сглаженный `Seasonal_Factor`


## Открытые методологические вопросы

1. Какой cutoff прогноза используем: начало дня, конец дня или прогноз на следующий день
2. Можно ли использовать M5 flow-признаки Росказны на текущую дату, или их надо лагировать под выбранный cutoff
3. Нужен ли календарь публикаций ЦБ для точного point-in-time лага M1 и месячных budget-данных
4. Как именно устраняем двойной счет налоговой сезонности: residualization, условные веса или отдельная регуляризация агрегатора
5. Будет ли финальная модель использовать все component features или сначала модульные stress-сигналы
